<a href="https://colab.research.google.com/github/BarisBayazit/Thermal-Plant-Allocation-Project/blob/main/Power-Plant-Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install folium openpyxl

import pandas as pd
import folium
import math
from google.colab import files


In [11]:
uploaded = files.upload()  # A file picker will appear — upload your Excel file here

Saving Plant_Data.xlsx to Plant_Data (1).xlsx


In [12]:
df = pd.read_excel("Plant_Data.xlsx")

# Keep only the relevant columns and rename them
df = df.iloc[:, :4]
df.columns = ["plant_name", "lat", "lon", "category"]

# Drop any rows where plant name is missing (empty rows in Excel)
df = df.dropna(subset=["plant_name"]).reset_index(drop=True)

print(f"Loaded {len(df)} plants")
print(df.head())


Loaded 33 plants
  plant_name      lat     lon category
0    Afşin A  38.2470  36.933     Coal
1    Afşin B  38.2475  36.945     Coal
2  Seyitömer  39.4245  29.259     Coal
3  Tunçbilek  39.5490  29.606     Coal
4  Çatalağzı  41.4520  31.761     Coal


In [14]:
facilities = {
    "Eskişehir": {"lat": 39.8379, "lon": 30.3056},
    "Mersin":    {"lat": 36.8697, "lon": 34.7668}
}


In [15]:
def haversine(lat1, lon1, lat2, lon2):
    """
    Calculate the great-circle distance in km between two points
    on Earth using the Haversine formula.
    More accurate than straight-line Euclidean distance,
    especially across large geographic areas like Turkey.
    """
    R = 6371  # Earth's radius in km

    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))

    return round(R * c, 2)


In [16]:
for facility_name, coords in facilities.items():
    df[f"dist_{facility_name}"] = df.apply(
        lambda row: haversine(row["lat"], row["lon"],
                              coords["lat"], coords["lon"]),
        axis=1
    )

# Assign each plant to its nearest facility
df["assigned_facility"] = df.apply(
    lambda row: min(facilities.keys(),
                    key=lambda f: row[f"dist_{f}"]),
    axis=1
)

df["min_distance_km"] = df.apply(
    lambda row: row[f"dist_{row['assigned_facility']}"],
    axis=1
)

print(df[["plant_name", "dist_Eskişehir", "dist_Mersin", "assigned_facility", "min_distance_km"]])

                               plant_name  dist_Eskişehir  dist_Mersin  \
0                                 Afşin A          598.90       244.76   
1                                 Afşin B          599.87       245.62   
2                               Seyitömer          100.73       559.02   
3                               Tunçbilek           67.93       540.30   
4                               Çatalağzı          217.46       571.55   
5                                Orhaneli          112.51       605.82   
6                                  Kangal          611.15       349.12   
7                           18 Mart (Çan)          278.32       758.01   
8                                Soma A-B          242.44       677.50   
9                                 Yatağan          335.30       589.52   
10                                Yeniköy          357.30       615.39   
11                               Kemerköy          357.14       616.21   
12                       İskenderun At

In [17]:
print("\n--- Allocation Summary ---")
print(df["assigned_facility"].value_counts())

print("\n--- By Fuel Type ---")
print(df.groupby(["assigned_facility", "category"])["plant_name"].count())

print(f"\nAverage distance to assigned facility: {df['min_distance_km'].mean():.1f} km")
print(f"Furthest plant: {df.loc[df['min_distance_km'].idxmax(), 'plant_name']} ({df['min_distance_km'].max()} km)")
print(f"Closest plant:  {df.loc[df['min_distance_km'].idxmin(), 'plant_name']} ({df['min_distance_km'].min()} km)")



--- Allocation Summary ---
assigned_facility
Eskişehir    25
Mersin        8
Name: count, dtype: int64

--- By Fuel Type ---
assigned_facility  category   
Eskişehir          Coal           19
                   Gas             5
                   Liquid Fuel     1
Mersin             Coal            7
                   Liquid Fuel     1
Name: plant_name, dtype: int64

Average distance to assigned facility: 218.2 km
Furthest plant: Hopa Fuel (762.45 km)
Closest plant:  Tunçbilek (67.93 km)


In [18]:
# Color each plant by assigned facility
facility_colors = {
    "Eskişehir": "blue",
    "Mersin": "green"
}

# Icon shape by fuel type
fuel_icons = {
    "Coal":      "industry",
    "Gas":   "fire",
    "Liquid Fuel": "tint"
}

# Center map on Turkey
m = folium.Map(location=[39.0, 32.0], zoom_start=6)

# Plot each thermal plant
for _, row in df.iterrows():
    color  = facility_colors.get(row["assigned_facility"], "gray")
    icon   = fuel_icons.get(row["category"], "circle")

    folium.Marker(
        location=[row["lat"], row["lon"]],
        popup=folium.Popup(
            f"""
            <b>{row['plant_name']}</b><br>
            Fuel Type: {row['category']}<br>
            Assigned to: {row['assigned_facility']}<br>
            Distance: {row['min_distance_km']} km
            """,
            max_width=200
        ),
        tooltip=row["plant_name"],
        icon=folium.Icon(color=color, icon=icon, prefix="fa")
    ).add_to(m)

# Plot the two production facilities
for facility_name, coords in facilities.items():
    folium.Marker(
        location=[coords["lat"], coords["lon"]],
        popup=folium.Popup(f"<b>Çimsa {facility_name}</b>", max_width=150),
        tooltip=f"Facility: {facility_name}",
        icon=folium.Icon(color="red", icon="star", prefix="fa")
    ).add_to(m)

# Add a legend
legend_html = """
<div style="position: fixed; bottom: 40px; left: 40px; z-index: 1000;
            background-color: white; padding: 12px; border-radius: 8px;
            border: 1px solid #ccc; font-size: 13px;">
    <b>Assigned Facility</b><br>
    <i class="fa fa-map-marker" style="color:blue"></i> Eskişehir<br>
    <i class="fa fa-map-marker" style="color:green"></i> Mersin<br>
    <i class="fa fa-map-marker" style="color:red"></i> Production Facility<br>
    <br>
    <b>Fuel Type</b><br>
    <i class="fa fa-industry"></i> Coal<br>
    <i class="fa fa-fire"></i> Gas<br>
    <i class="fa fa-tint"></i> Liquid Fuel
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m  # Display the map inline in Colab


In [19]:
m.save("supplier_map.html")
print("Map saved as supplier_map.html")


Map saved as supplier_map.html


In [20]:
from google.colab import files
files.download("supplier_map.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>